# Automated Multi-Modal X-Ray Fracture & Crack Reporting System

**Project type:** Deep Learning course project + Streamlit Cloud deployment prototype.  
**Main objective:** detect X-ray fracture/crack suspicion, show suspicious visual evidence with Grad-CAM, and generate an interpretable risk report from image + patient factors.

**Design guarantees**

- CNN-only vision model. No Vision Transformer or transformer-based image architecture is used.
- TensorFlow/Keras is the primary framework for training, saving, and inference.
- FracAtlas is downloaded automatically inside the notebook with `kagglehub`.
- No dataset files are embedded in the notebook or expected in GitHub.
- The final trained model is saved as `fracture_model.keras`.
- The Streamlit app loads the saved `.keras` model and performs inference only.
- The notebook also includes an in-notebook Gradio GUI for Colab demonstration after training.

**Clinical safety note:** this is an academic decision-support prototype. It is not a medical device and must not replace radiologist or physician review.

**Reference:** Abedeen et al., FracAtlas: A Dataset for Fracture Classification, Localization and Segmentation of Musculoskeletal Radiographs. Scientific Data, 2023. https://doi.org/10.1038/s41597-023-02432-4

## 1. Runtime Setup

This cell installs only the libraries needed for the notebook, imports dependencies, enables reproducibility settings, and checks GPU availability. The notebook is designed for Google Colab GPU runtime and should be runnable from top to bottom without manual file upload prompts.

In [ ]:
%pip install -q kagglehub opencv-python-headless seaborn tqdm scikit-learn pycocotools gradio

import gc
import json
import os
import random
import shutil
import sys
import warnings
from pathlib import Path
from zipfile import ZipFile

import cv2
import kagglehub
import gradio as gr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image, ImageFile
from pycocotools.coco import COCO
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.keras.utils.set_random_seed(SEED)
except Exception:
    pass

for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except Exception:
        pass

print('TensorFlow version:', tf.__version__)
print('Python version:', sys.version.split()[0])
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Interpretation: if a GPU is listed, the notebook is ready for accelerated CNN training.')

## 2. Project Configuration

These constants control runtime cost and model strength. The defaults train the full dataset with a Colab-safe EfficientNetB0 CNN. For a quick classroom test, set `MAX_IMAGES_PER_CLASS` to a small number such as `400`; for final submission, keep it as `None`.

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
WORK_ROOT = Path('/content/fracatlas_work') if IN_COLAB else Path('fracatlas_work')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

FRACATLAS_DATASET = 'mahmudulhasantasin/fracatlas-original-dataset'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

SUPPORTED_BACKBONES = [
    'EfficientNetB0',
    'EfficientNetB3',
    'DenseNet121',
    'ResNet50',
    'MobileNetV2',
    'InceptionV3',
    'Xception',
]
CNN_BACKBONE_NAME = 'DenseNet121'    # Strong medical X-ray default. Change to any name in SUPPORTED_BACKBONES.
RUN_BACKBONE_COMPARISON = False      # Set True to run a small comparison between all supported CNN backbones.
FINAL_BACKBONE_FROM_COMPARISON = True
COMPARISON_IMAGES_PER_CLASS = 250    # Small subset per class for fast comparison.
COMPARISON_EPOCHS = 2

MAX_IMAGES_PER_CLASS = None          # Example quick run: 400. Final run: None.
CREATE_FOLDER_SPLIT = True           # Creates train/val/test class folders using links when possible.
EPOCHS_HEAD = 10
EPOCHS_FINE = 8
FINE_TUNE = True
FINE_TUNE_LAST_N_LAYERS = 30

IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
EXCLUDE_IMAGE_HINTS = ('mask', 'segmentation', 'annotation', 'labelme', 'bbox', 'bounding_box')
POSITIVE_HINTS = ('fracture', 'fractured', 'crack', 'broken')
NEGATIVE_HINTS = (
    'non fractured', 'non fracture', 'nonfractured', 'non-fractured', 'non_fractured',
    'not fractured', 'no fracture', 'normal', 'healthy', 'negative'
)

if CNN_BACKBONE_NAME not in SUPPORTED_BACKBONES:
    raise ValueError(f'CNN_BACKBONE_NAME must be one of: {SUPPORTED_BACKBONES}')

print('Work root:', WORK_ROOT)
print('Supported CNN backbones:', SUPPORTED_BACKBONES)
print('Selected final CNN backbone:', CNN_BACKBONE_NAME)
print('Run backbone comparison:', RUN_BACKBONE_COMPARISON)
print('Full dataset mode:', MAX_IMAGES_PER_CLASS is None)
print('Interpretation: these settings balance academic quality with Colab runtime limits.')

## 3. Automatic FracAtlas Download

This cell downloads the FracAtlas original dataset directly with `kagglehub`. It does not ask for uploaded files. The returned cache path is used as the raw dataset root for all later steps.

In [ ]:
print('Downloading FracAtlas dataset with kagglehub...')
dataset_path = kagglehub.dataset_download(FRACATLAS_DATASET)
RAW_DIR = Path(dataset_path)
DATA_ROOT = WORK_ROOT
selected_source = FRACATLAS_DATASET

file_count = sum(1 for p in RAW_DIR.rglob('*') if p.is_file())
total_mb = sum(p.stat().st_size for p in RAW_DIR.rglob('*') if p.is_file()) / (1024 ** 2)

if file_count == 0:
    raise RuntimeError('FracAtlas download completed, but no files were found. Check dataset availability.')

print('Selected data source:', selected_source)
print('Dataset path:', RAW_DIR)
print(f'Files discovered: {file_count:,}')
print(f'Approximate extracted size: {total_mb:.1f} MB')
print('Interpretation: FracAtlas is now local to this Colab runtime and remains outside the project repository.')

## 4. COCO Annotation Preview

FracAtlas includes COCO-style fracture masks in many releases. This section previews one fracture mask overlay for dataset understanding. The model trained below is still a classifier with Grad-CAM localization, not a segmentation model.

In [ ]:
def find_coco_annotation_file(root):
    json_files = sorted(Path(root).rglob('*.json'))
    preferred = []
    fallback = []
    for json_path in json_files:
        lower = str(json_path).lower()
        if 'coco' in lower and ('fracture' in lower or 'mask' in lower):
            preferred.append(json_path)
        elif 'coco' in lower:
            fallback.append(json_path)
    return preferred[0] if preferred else (fallback[0] if fallback else None)


def find_image_by_filename(root, file_name):
    file_name = Path(file_name).name
    matches = list(Path(root).rglob(file_name))
    if matches:
        return matches[0]
    stem = Path(file_name).stem
    for ext in IMAGE_EXTENSIONS:
        matches = list(Path(root).rglob(stem + ext))
        if matches:
            return matches[0]
    return None

coco_json_path = find_coco_annotation_file(RAW_DIR)
if coco_json_path is None:
    print('No COCO annotation JSON was found in this dataset copy. Continuing with classification labels only.')
else:
    print('COCO annotation file:', coco_json_path)
    coco = COCO(str(coco_json_path))
    image_ids = sorted(coco.getImgIds())
    if not image_ids:
        print('The COCO file loaded, but it contains no image IDs.')
    else:
        image_id = image_ids[min(17, len(image_ids) - 1)]
        img_info = coco.loadImgs([image_id])[0]
        image_path = find_image_by_filename(RAW_DIR, img_info['file_name'])
        print('Selected COCO image id:', image_id)
        print('Selected file:', img_info['file_name'])
        print('Resolved path:', image_path)

        if image_path is not None:
            image = np.asarray(Image.open(image_path).convert('RGB'))
            ann_ids = coco.getAnnIds(imgIds=[image_id])
            anns = coco.loadAnns(ann_ids)
            mask = np.zeros(image.shape[:2], dtype=np.float32)
            for ann in anns:
                mask = np.maximum(mask, coco.annToMask(ann).astype(np.float32))

            plt.figure(figsize=(12, 4))
            ax = plt.subplot(1, 3, 1)
            ax.imshow(image)
            ax.set_title('FracAtlas X-ray')
            ax.axis('off')
            ax = plt.subplot(1, 3, 2)
            ax.imshow(mask, cmap='Reds')
            ax.set_title('COCO fracture mask')
            ax.axis('off')
            ax = plt.subplot(1, 3, 3)
            ax.imshow(image)
            ax.imshow(mask, cmap='jet', alpha=0.35)
            ax.set_title('Mask overlay')
            ax.axis('off')
            plt.tight_layout()
            plt.show()
            print('Interpretation: the mask overlay documents annotated fracture regions available in FracAtlas.')
        else:
            print('The referenced COCO image file was not found in the extracted dataset.')

## 5. Dataset Discovery and Binary Label Construction

This cell creates a robust binary dataframe with `0 = Normal/Non-fractured` and `1 = Fracture/Crack`. It prioritizes trusted folder names such as `Fractured` and `Non_fractured`, then falls back to CSV metadata if needed. It also validates image files to avoid training crashes from corrupt inputs.

In [ ]:
def collect_image_paths(root):
    paths = []
    for path in Path(root).rglob('*'):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            lower = str(path).lower()
            if any(hint in lower for hint in EXCLUDE_IMAGE_HINTS):
                continue
            paths.append(path)
    return sorted(paths)


def normalize_text(value):
    return str(value).strip().lower().replace('_', ' ').replace('-', ' ')


def normalize_label(value):
    if pd.isna(value):
        return None
    text = normalize_text(value)
    if text in {'1', 'true', 'yes', 'positive', 'fracture', 'fractured', 'crack', 'broken'}:
        return 1
    if text in {'0', 'false', 'no', 'negative', 'normal', 'healthy'}:
        return 0
    if any(hint.replace('_', ' ').replace('-', ' ') in text for hint in NEGATIVE_HINTS):
        return 0
    if any(hint in text for hint in POSITIVE_HINTS):
        return 1
    if 'no finding' in text:
        return 0
    return None


def infer_label_from_path(path):
    normalized_parts = [normalize_text(part) for part in path.parts]
    joined = ' '.join(normalized_parts)
    if any(hint.replace('_', ' ').replace('-', ' ') in joined for hint in NEGATIVE_HINTS):
        return 0
    if any(hint in joined for hint in POSITIVE_HINTS):
        return 1
    return None


def likely_path_columns(columns):
    candidates = []
    hints = ('image index', 'image id', 'image name', 'image_name', 'filename', 'file name', 'file_name', 'path')
    for col in columns:
        clean = normalize_text(col)
        if any(hint.replace('_', ' ') in clean for hint in hints) and 'label' not in clean and 'finding' not in clean:
            candidates.append(col)
    return candidates


def likely_label_columns(columns):
    candidates = []
    hints = ('fracture', 'fractured', 'label', 'class', 'category', 'target', 'finding', 'status')
    for col in columns:
        clean = normalize_text(col)
        if any(hint in clean for hint in hints):
            candidates.append(col)
    return candidates


def is_valid_image(path):
    try:
        with Image.open(path) as img:
            img.verify()

        image_bytes = tf.io.read_file(str(path))
        image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        _ = tf.image.resize(tf.cast(image, tf.float32), IMG_SIZE).numpy()
        return True
    except Exception:
        return False

image_paths = collect_image_paths(RAW_DIR)
print(f'Candidate X-ray image files discovered: {len(image_paths):,}')

folder_records = []
for path in image_paths:
    label = infer_label_from_path(path)
    if label is not None:
        folder_records.append({'image_path': str(path), 'label': int(label), 'label_source': 'folder_name'})
folder_df = pd.DataFrame(folder_records).drop_duplicates('image_path') if folder_records else pd.DataFrame()

name_to_path = {p.name: p for p in image_paths}
stem_to_path = {p.stem: p for p in image_paths}
csv_records = []
for csv_path in sorted(RAW_DIR.rglob('*.csv')):
    try:
        meta = pd.read_csv(csv_path)
    except Exception:
        continue
    path_cols = likely_path_columns(meta.columns)
    label_cols = likely_label_columns(meta.columns)
    if not path_cols or not label_cols:
        continue
    for path_col in path_cols:
        for label_col in label_cols:
            for _, row in meta.iterrows():
                raw_name = str(row.get(path_col, '')).strip()
                label = normalize_label(row.get(label_col))
                if not raw_name or label is None:
                    continue
                key_name = Path(raw_name).name
                key_stem = Path(raw_name).stem
                image_path = name_to_path.get(key_name) or stem_to_path.get(key_stem)
                if image_path is not None:
                    csv_records.append({'image_path': str(image_path), 'label': int(label), 'label_source': csv_path.name})
csv_df = pd.DataFrame(csv_records).drop_duplicates('image_path') if csv_records else pd.DataFrame()

if not folder_df.empty and folder_df['label'].nunique() == 2:
    image_df = folder_df.copy()
    source_used = 'folder names'
elif not csv_df.empty and csv_df['label'].nunique() == 2:
    image_df = csv_df.copy()
    source_used = 'CSV metadata'
else:
    raise ValueError(
        f'Could not construct both classes. Folder rows={len(folder_df)}, CSV rows={len(csv_df)}. Inspect RAW_DIR: {RAW_DIR}'
    )

print('Validating image files with PIL and TensorFlow decoders...')
valid_mask = []
invalid_image_paths = []
for image_path in tqdm(image_df['image_path'], total=len(image_df)):
    ok = is_valid_image(image_path)
    valid_mask.append(ok)
    if not ok:
        invalid_image_paths.append(image_path)

if invalid_image_paths:
    print(f'Removed invalid image files: {len(invalid_image_paths)}')
    display(pd.DataFrame({'invalid_image_path': invalid_image_paths}).head(20))
else:
    print('No invalid images found by strict validation.')

image_df = image_df.loc[valid_mask].drop_duplicates('image_path').reset_index(drop=True)

if image_df['label'].nunique() < 2:
    raise ValueError('After image validation, only one class remains. Check dataset parsing.')

image_df['label_name'] = image_df['label'].map({0: 'Normal/Non-fractured', 1: 'Fracture/Crack'})
image_df = image_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

print('Label source used:', source_used)
print(image_df['label_name'].value_counts())
display(image_df.head())
print('Interpretation: each row now maps a validated X-ray image to a binary fracture/crack label.')

## 6. Dataset Audit

A serious medical AI notebook should not train blindly. This audit reports class balance and checks representative image dimensions. It helps identify label imbalance and unexpected non-radiograph files before training begins.

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=image_df, x='label_name', order=['Normal/Non-fractured', 'Fracture/Crack'])
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Image Count')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

sample_for_shape = image_df.sample(min(80, len(image_df)), random_state=SEED)
shape_rows = []
for _, row in sample_for_shape.iterrows():
    with Image.open(row['image_path']) as img:
        shape_rows.append({'width': img.width, 'height': img.height, 'label_name': row['label_name']})
shape_df = pd.DataFrame(shape_rows)
display(shape_df.describe())
print('Interpretation: class balance informs class weighting, while dimensions confirm that resizing to 224x224 is reasonable for CNN training.')

## 7. Optional Local CSV Inspection

Your workspace contains `Data_Entry_2017_v2020 (1).csv`, which is NIH ChestX-ray14 metadata. It is not FracAtlas and should not be mixed into the fracture image labels. This optional cell only inspects the file if it already exists in the runtime.

In [ ]:
OPTIONAL_CSV = Path('/content/Data_Entry_2017_v2020 (1).csv') if IN_COLAB else Path('Data_Entry_2017_v2020 (1).csv')
if OPTIONAL_CSV.exists():
    optional_df = pd.read_csv(OPTIONAL_CSV)
    print('Optional CSV found:', OPTIONAL_CSV)
    print('Shape:', optional_df.shape)
    print('Columns:', list(optional_df.columns))
    display(optional_df.head())
    print('Interpretation: this is chest X-ray metadata, so it is documented but not used for FracAtlas fracture training.')
else:
    print('Optional NIH metadata CSV was not found. This is fine; the fracture model uses FracAtlas plus generated patient factors.')

## 8. Synthetic Patient Tabular Data

FracAtlas does not provide the requested patient history fields. To demonstrate a multi-modal architecture reproducibly, this cell generates synthetic patient factors with clinically plausible relationships. Patient name is never used as a model feature; it is only used in report text.

In [ ]:
FEATURE_NAMES = [
    'age_norm', 'gender_male', 'diabetes', 'osteoporosis', 'smoking', 'previous_fracture', 'pain_norm',
    'injury_arm', 'injury_leg', 'injury_spine', 'injury_rib', 'injury_hip',
    'treatment_cast', 'treatment_surgery', 'treatment_physiotherapy', 'treatment_medication', 'treatment_none',
]
INJURY_LOCATIONS = ['Arm', 'Leg', 'Spine', 'Rib', 'Hip']
rng = np.random.default_rng(SEED)


def generate_patient_record(label):
    age = int(np.clip(rng.normal(42 + 13 * label, 18), 5, 95))
    gender = rng.choice(['Male', 'Female'])
    diabetes = bool(rng.random() < (0.09 + 0.07 * (age >= 60)))
    osteoporosis = bool(rng.random() < (0.04 + 0.20 * (age >= 60) + 0.05 * label))
    smoking = bool(rng.random() < 0.21)
    previous_fracture = bool(rng.random() < (0.09 + 0.10 * label))
    pain_level = int(np.clip(round(rng.normal(4.0 + 2.2 * label, 1.8)), 1, 10))
    injury_location = rng.choice(INJURY_LOCATIONS, p=[0.34, 0.30, 0.10, 0.11, 0.15])
    if label == 1:
        treatment_history = rng.choice(['none', 'cast', 'medication', 'physiotherapy', 'surgery'], p=[0.30, 0.27, 0.19, 0.13, 0.11])
    else:
        treatment_history = rng.choice(['none', 'medication', 'physiotherapy'], p=[0.74, 0.19, 0.07])
    return {
        'patient_name': 'Synthetic Patient',
        'age': age,
        'gender': gender,
        'diabetes': diabetes,
        'osteoporosis': osteoporosis,
        'smoking': smoking,
        'previous_fracture': previous_fracture,
        'pain_level': pain_level,
        'injury_location': injury_location,
        'treatment_history': treatment_history,
    }


def encode_patient_row(row):
    text = str(row.get('treatment_history', '')).lower()
    location = str(row.get('injury_location', '')).lower()
    age_norm = np.clip(float(row.get('age', 0)), 0.0, 110.0) / 110.0
    pain_norm = np.clip(float(row.get('pain_level', 1)), 1.0, 10.0) / 10.0
    values = [
        age_norm,
        1.0 if row.get('gender') == 'Male' else 0.0,
        float(bool(row.get('diabetes'))),
        float(bool(row.get('osteoporosis'))),
        float(bool(row.get('smoking'))),
        float(bool(row.get('previous_fracture'))),
        pain_norm,
        1.0 if location == 'arm' else 0.0,
        1.0 if location == 'leg' else 0.0,
        1.0 if location == 'spine' else 0.0,
        1.0 if location == 'rib' else 0.0,
        1.0 if location == 'hip' else 0.0,
        1.0 if 'cast' in text else 0.0,
        1.0 if 'surgery' in text or 'operation' in text else 0.0,
        1.0 if 'physio' in text or 'rehab' in text else 0.0,
        1.0 if 'medication' in text or 'analgesic' in text or 'painkiller' in text else 0.0,
        1.0 if not text.strip() or 'none' in text or 'no treatment' in text else 0.0,
    ]
    return pd.Series(values, index=FEATURE_NAMES, dtype='float32')

patient_df = pd.DataFrame([generate_patient_record(label) for label in image_df['label'].values])
model_df = pd.concat([image_df.reset_index(drop=True), patient_df], axis=1)
encoded_features = model_df.apply(encode_patient_row, axis=1)
model_df = pd.concat([model_df, encoded_features], axis=1)

print('Model dataframe shape:', model_df.shape)
print('Tabular feature count:', len(FEATURE_NAMES))
display(model_df[['image_path', 'label_name', 'age', 'gender', 'pain_level', 'injury_location', 'treatment_history'] + FEATURE_NAMES[:5]].head())
print('Interpretation: every X-ray now has a synchronized structured patient vector for multi-modal training.')

## 9. Train/Validation/Test Split and Class Weights

The split is stratified to preserve class proportions. Class weights are computed from the training fold to reduce bias toward the majority class.

In [ ]:
def remove_duplicate_columns(frame):
    return frame.loc[:, ~frame.columns.duplicated()].copy()

model_df = remove_duplicate_columns(model_df)
working_df = model_df.copy()
if MAX_IMAGES_PER_CLASS is not None:
    working_df = (
        working_df.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), MAX_IMAGES_PER_CLASS), random_state=SEED))
        .reset_index(drop=True)
    )

train_df, temp_df = train_test_split(
    working_df,
    test_size=0.30,
    stratify=working_df['label'],
    random_state=SEED,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=SEED,
)

train_df = remove_duplicate_columns(train_df)
val_df = remove_duplicate_columns(val_df)
test_df = remove_duplicate_columns(test_df)

classes = np.array([0, 1])
weights = compute_class_weight(class_weight='balanced', classes=classes, y=train_df['label'].values)
class_weight = {int(cls): float(weight) for cls, weight in zip(classes, weights)}

print('Train size:', len(train_df), train_df['label_name'].value_counts().to_dict())
print('Validation size:', len(val_df), val_df['label_name'].value_counts().to_dict())
print('Test size:', len(test_df), test_df['label_name'].value_counts().to_dict())
print('Class weights:', class_weight)
print('Duplicate columns after split:', {
    'train': int(train_df.columns.duplicated().sum()),
    'val': int(val_df.columns.duplicated().sum()),
    'test': int(test_df.columns.duplicated().sum()),
})
print('Interpretation: validation and test are held out until model selection/evaluation.')

## 10. Kaggle-Style Folder Split for Inspection

This mirrors the common Kaggle workflow with `train`, `val`, and `test` folders containing `fractured` and `non_fractured` subfolders. To avoid duplicating a large dataset, the cell uses symlinks when possible and copies only as a fallback.

In [ ]:
SPLIT_DIR = DATA_ROOT / 'keras_folder_split'
CLASS_DIR_NAMES = {0: 'non_fractured', 1: 'fractured'}


def link_or_copy(source, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() or destination.is_symlink():
        return 'exists'
    try:
        os.symlink(source, destination)
        return 'linked'
    except Exception:
        shutil.copy2(source, destination)
        return 'copied'


def create_split_folders(frame, split_name):
    counts = {'linked': 0, 'copied': 0, 'exists': 0}
    for _, row in tqdm(frame.iterrows(), total=len(frame), desc=f'Preparing {split_name} folders'):
        source = Path(row['image_path'])
        class_dir = CLASS_DIR_NAMES[int(row['label'])]
        destination = SPLIT_DIR / split_name / class_dir / source.name
        status = link_or_copy(source, destination)
        counts[status] += 1
    return counts

if CREATE_FOLDER_SPLIT:
    split_counts = {
        'train': create_split_folders(train_df, 'train'),
        'val': create_split_folders(val_df, 'val'),
        'test': create_split_folders(test_df, 'test'),
    }
    print('Folder split root:', SPLIT_DIR)
    print('Link/copy summary:', split_counts)
    for split_name in ['train', 'val', 'test']:
        for class_name in ['fractured', 'non_fractured']:
            count = len(list((SPLIT_DIR / split_name / class_name).glob('*')))
            print(f'{split_name}/{class_name}: {count}')
else:
    print('CREATE_FOLDER_SPLIT=False, so folder split creation was skipped.')
print('Interpretation: this folder layout is for inspection and baseline compatibility; the multi-modal model trains from the dataframe.')

## 11. TensorFlow Data Pipeline

The data pipeline decodes images, resizes them to `224 x 224`, and pairs them with tabular patient vectors. Inputs are returned as a dictionary with names matching the Keras model inputs.

In [ ]:
def decode_image(path):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMG_SIZE)
    return tf.cast(image, tf.float32)


def make_dataset(frame, shuffle=False, batch_size=BATCH_SIZE):
    frame = remove_duplicate_columns(frame)
    missing = [col for col in FEATURE_NAMES if col not in frame.columns]
    if missing:
        raise ValueError(f'Missing feature columns: {missing}')

    paths = frame['image_path'].astype(str).values
    tabular = frame.loc[:, FEATURE_NAMES].astype('float32').to_numpy()
    labels = frame['label'].astype('float32').values

    if tabular.shape[1] != len(FEATURE_NAMES):
        raise ValueError(f'Expected {len(FEATURE_NAMES)} patient features, found {tabular.shape[1]}')

    dataset = tf.data.Dataset.from_tensor_slices((paths, tabular, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(frame), seed=SEED, reshuffle_each_iteration=True)

    def load_example(path, patient_features, label):
        image = decode_image(path)
        patient_features = tf.ensure_shape(patient_features, [len(FEATURE_NAMES)])
        inputs = {'xray_image': image, 'patient_features': patient_features}
        return inputs, tf.expand_dims(label, axis=-1)

    dataset = dataset.map(load_example, num_parallel_calls=AUTOTUNE)
    dataset = dataset.apply(tf.data.experimental.ignore_errors())
    return dataset.batch(batch_size).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

for inputs, labels in train_ds.take(1):
    print('Image batch shape:', inputs['xray_image'].shape)
    print('Patient feature batch shape:', inputs['patient_features'].shape)
    print('Label batch shape:', labels.shape)
print('Interpretation: the model will receive synchronized image and patient-feature batches with exactly the configured patient features.')

## 12. Visual Quality Check

This visualization checks that the model is seeing radiographs with correct labels rather than masks, annotation thumbnails, or corrupted files.

In [ ]:
plt.figure(figsize=(12, 8))
for inputs, labels in train_ds.take(1):
    images = inputs['xray_image'].numpy()
    y = labels.numpy().reshape(-1).astype(int)
    for i in range(min(9, len(images))):
        ax = plt.subplot(3, 3, i + 1)
        ax.imshow(np.clip(images[i] / 255.0, 0, 1))
        ax.set_title('Fracture/Crack' if y[i] == 1 else 'Normal/Non-fractured')
        ax.axis('off')
plt.tight_layout()
plt.show()
print('Interpretation: visually verify that labels and images look reasonable before training.')

## 13. CNN Multi-Modal Model Architecture

The notebook now supports several strong CNN backbones: EfficientNet, DenseNet, ResNet, MobileNet, Inception, and Xception. All are convolutional models, not Vision Transformers. The image branch is fused with a dense patient-feature branch, and the named `last_conv_features` layer supports Grad-CAM localization.

In [ ]:
from tensorflow.keras import Model, layers, regularizers

BACKBONE_BUILDERS = {
    'efficientnetb0': tf.keras.applications.EfficientNetB0,
    'efficientnetb3': tf.keras.applications.EfficientNetB3,
    'densenet121': tf.keras.applications.DenseNet121,
    'resnet50': tf.keras.applications.ResNet50,
    'mobilenetv2': tf.keras.applications.MobileNetV2,
    'inceptionv3': tf.keras.applications.InceptionV3,
    'xception': tf.keras.applications.Xception,
}


def canonical_backbone_name(name):
    normalized = str(name).replace('_', '').replace('-', '').lower()
    for supported in SUPPORTED_BACKBONES:
        if supported.lower() == normalized:
            return supported
    raise ValueError(f'Unsupported backbone {name}. Choose from {SUPPORTED_BACKBONES}')


def add_backbone_preprocessing(x, backbone_name):
    name = canonical_backbone_name(backbone_name).lower()
    if name.startswith('efficientnet'):
        return x
    if name in {'mobilenetv2', 'inceptionv3', 'xception'}:
        return layers.Rescaling(1.0 / 127.5, offset=-1.0, name=f'{name}_rescale_minus1_1')(x)
    if name == 'densenet121':
        x = layers.Rescaling(1.0 / 255.0, name='densenet_rescale_0_1')(x)
        return layers.Normalization(
            mean=[0.485, 0.456, 0.406],
            variance=[0.229 ** 2, 0.224 ** 2, 0.225 ** 2],
            name='densenet_imagenet_normalization',
        )(x)
    if name == 'resnet50':
        return layers.Normalization(
            mean=[123.68, 116.779, 103.939],
            variance=[1.0, 1.0, 1.0],
            name='resnet_imagenet_mean_centering',
        )(x)
    return x


def build_cnn_backbone(name):
    canonical = canonical_backbone_name(name)
    key = canonical.lower()
    builder = BACKBONE_BUILDERS[key]
    kwargs = {
        'input_shape': (IMG_SIZE[0], IMG_SIZE[1], 3),
        'include_top': False,
        'weights': 'imagenet',
    }
    if key == 'mobilenetv2':
        kwargs['alpha'] = 1.0
    return builder(**kwargs)


def build_multimodal_cnn(num_tabular_features, backbone_name=CNN_BACKBONE_NAME):
    backbone_name = canonical_backbone_name(backbone_name)

    image_input = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name='xray_image')
    x = layers.RandomFlip('horizontal', name='aug_flip')(image_input)
    x = layers.RandomRotation(0.04, name='aug_rotation')(x)
    x = layers.RandomZoom(0.08, name='aug_zoom')(x)
    x = layers.RandomContrast(0.10, name='aug_contrast')(x)
    x = add_backbone_preprocessing(x, backbone_name)

    backbone = build_cnn_backbone(backbone_name)
    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.Activation('linear', name='last_conv_features')(x)
    x = layers.GlobalAveragePooling2D(name='image_global_average_pooling')(x)
    x = layers.Dropout(0.30, name='image_dropout')(x)
    image_features = layers.Dense(128, activation='relu', name='image_embedding')(x)

    patient_input = layers.Input(shape=(num_tabular_features,), name='patient_features')
    t = layers.BatchNormalization(name='patient_batch_norm')(patient_input)
    t = layers.Dense(64, activation='relu', name='patient_dense_1')(t)
    t = layers.Dropout(0.20, name='patient_dropout_1')(t)
    patient_features = layers.Dense(32, activation='relu', name='patient_embedding')(t)

    fused = layers.Concatenate(name='multimodal_fusion')([image_features, patient_features])
    fused = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4), name='fusion_dense_1')(fused)
    fused = layers.Dropout(0.35, name='fusion_dropout_1')(fused)
    fused = layers.Dense(64, activation='relu', name='fusion_dense_2')(fused)
    output = layers.Dense(1, activation='sigmoid', name='fracture_probability')(fused)

    model = Model(inputs=[image_input, patient_input], outputs=output, name='cnn_multimodal_fracture_model')
    return model, backbone


def compile_model(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.AUC(name='auc'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
        ],
    )

model, backbone = build_multimodal_cnn(num_tabular_features=len(FEATURE_NAMES), backbone_name=CNN_BACKBONE_NAME)
compile_model(model, learning_rate=1e-3)
model.summary()
print('Supported CNN backbones:', SUPPORTED_BACKBONES)
print('CNN backbone selected:', CNN_BACKBONE_NAME)
print('Interpretation: this is a CNN-only multi-modal classifier with a Grad-CAM-compatible feature layer.')

## 13.1 Optional CNN Backbone Comparison

This optional experiment trains each supported CNN backbone briefly on a balanced subset and compares validation AUC. It is disabled by default to keep `Run all` practical. For the final academic report, you can enable `RUN_BACKBONE_COMPARISON = True` in configuration, record the comparison table, and let the notebook select the best backbone for the final full training run.

In [ ]:
backbone_comparison_df = pd.DataFrame()

if RUN_BACKBONE_COMPARISON:
    comparison_df = (
        train_df.groupby('label', group_keys=False)
        .apply(lambda x: x.sample(min(len(x), COMPARISON_IMAGES_PER_CLASS), random_state=SEED))
        .reset_index(drop=True)
    )
    comparison_train_df, comparison_val_df = train_test_split(
        comparison_df,
        test_size=0.25,
        stratify=comparison_df['label'],
        random_state=SEED,
    )
    comparison_train_ds = make_dataset(comparison_train_df, shuffle=True, batch_size=BATCH_SIZE)
    comparison_val_ds = make_dataset(comparison_val_df, shuffle=False, batch_size=BATCH_SIZE)

    comparison_rows = []
    for backbone_name in SUPPORTED_BACKBONES:
        print('\n' + '=' * 70)
        print('Testing CNN backbone:', backbone_name)
        tf.keras.backend.clear_session()
        candidate_model = None
        try:
            candidate_model, _ = build_multimodal_cnn(
                num_tabular_features=len(FEATURE_NAMES),
                backbone_name=backbone_name,
            )
            compile_model(candidate_model, learning_rate=1e-3)
            candidate_history = candidate_model.fit(
                comparison_train_ds,
                validation_data=comparison_val_ds,
                epochs=COMPARISON_EPOCHS,
                class_weight=class_weight,
                verbose=1,
            )
            best_val_auc = float(np.max(candidate_history.history.get('val_auc', [np.nan])))
            best_val_acc = float(np.max(candidate_history.history.get('val_accuracy', [np.nan])))
            params = int(candidate_model.count_params())
            comparison_rows.append({
                'backbone': backbone_name,
                'best_val_auc': best_val_auc,
                'best_val_accuracy': best_val_acc,
                'parameters': params,
                'status': 'ok',
            })
        except Exception as exc:
            comparison_rows.append({
                'backbone': backbone_name,
                'best_val_auc': np.nan,
                'best_val_accuracy': np.nan,
                'parameters': np.nan,
                'status': f'failed: {exc}',
            })
        finally:
            if candidate_model is not None:
                del candidate_model
            gc.collect()

    backbone_comparison_df = pd.DataFrame(comparison_rows).sort_values('best_val_auc', ascending=False)
    display(backbone_comparison_df)

    if FINAL_BACKBONE_FROM_COMPARISON and backbone_comparison_df['best_val_auc'].notna().any():
        CNN_BACKBONE_NAME = str(backbone_comparison_df.dropna(subset=['best_val_auc']).iloc[0]['backbone'])
        print('Selected final backbone from comparison:', CNN_BACKBONE_NAME)
    else:
        print('Keeping configured final backbone:', CNN_BACKBONE_NAME)
else:
    print('Backbone comparison skipped. Set RUN_BACKBONE_COMPARISON=True to compare:', SUPPORTED_BACKBONES)
    print('Selected final backbone:', CNN_BACKBONE_NAME)

# Rebuild the final model after optional comparison so training always uses the final selected backbone.
tf.keras.backend.clear_session()
model, backbone = build_multimodal_cnn(num_tabular_features=len(FEATURE_NAMES), backbone_name=CNN_BACKBONE_NAME)
compile_model(model, learning_rate=1e-3)
print('Final model backbone ready:', CNN_BACKBONE_NAME)

## 14. Training Phase 1: Frozen CNN Backbone

The first training phase freezes the pretrained CNN and learns the classifier head plus patient-feature branch. This is stable and reduces overfitting risk.

In [ ]:
MODEL_PATH = Path('fracture_model.keras')

# Ensure dataframes and datasets are clean immediately before training.
train_df = remove_duplicate_columns(train_df)
val_df = remove_duplicate_columns(val_df)
test_df = remove_duplicate_columns(test_df)
train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

for inputs, labels in train_ds.take(1):
    print('Training image batch:', inputs['xray_image'].shape)
    print('Training patient batch:', inputs['patient_features'].shape)
    print('Training label batch:', labels.shape)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_PATH),
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight,
    callbacks=callbacks,
)

print(f'Interpretation: phase 1 trained the {CNN_BACKBONE_NAME} decision layers while preserving pretrained CNN filters.')

## 15. Training Phase 2: Fine-Tuning

Fine-tuning adapts the last CNN layers to X-ray texture and bone morphology. The learning rate is intentionally low to avoid damaging pretrained filters.

In [ ]:
history_fine = None
if FINE_TUNE:
    backbone.trainable = True
    for layer in backbone.layers[:-FINE_TUNE_LAST_N_LAYERS]:
        layer.trainable = False
    compile_model(model, learning_rate=1e-5)
    history_fine = model.fit(
        train_ds,
        validation_data=val_ds,
        initial_epoch=len(history_head.epoch),
        epochs=len(history_head.epoch) + EPOCHS_FINE,
        class_weight=class_weight,
        callbacks=callbacks,
    )
else:
    print('Fine-tuning skipped by configuration.')

if MODEL_PATH.exists():
    model = tf.keras.models.load_model(MODEL_PATH)

gc.collect()
print('Saved model path:', MODEL_PATH.resolve())
print(f'Saved model size: {MODEL_PATH.stat().st_size / (1024 ** 2):.2f} MB')
print('Interpretation: the best validation-AUC model checkpoint is restored for evaluation.')

## 16. Training Curves

Learning curves support academic analysis by showing whether validation metrics improve consistently or diverge from training metrics.

In [ ]:
def history_to_frame(history, phase):
    if history is None:
        return pd.DataFrame()
    frame = pd.DataFrame(history.history)
    frame['epoch'] = np.arange(len(frame)) + 1
    frame['phase'] = phase
    return frame

history_df = pd.concat(
    [history_to_frame(history_head, 'frozen_head'), history_to_frame(history_fine, 'fine_tune')],
    ignore_index=True,
)

metrics_to_plot = [m for m in ['loss', 'auc', 'accuracy', 'precision', 'recall'] if m in history_df.columns]
plt.figure(figsize=(14, 3.5 * len(metrics_to_plot)))
for i, metric in enumerate(metrics_to_plot, start=1):
    ax = plt.subplot(len(metrics_to_plot), 1, i)
    ax.plot(history_df['epoch'], history_df[metric], marker='o', label=f'train_{metric}')
    val_metric = f'val_{metric}'
    if val_metric in history_df.columns:
        ax.plot(history_df['epoch'], history_df[val_metric], marker='o', label=val_metric)
    ax.set_title(metric)
    ax.set_xlabel('Epoch')
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.show()
print('Interpretation: validation AUC and validation loss are the main signals for model selection.')

## 17. Threshold Selection and Test Evaluation

The validation set is used to select a probability threshold by F1 score. The final held-out test set is then evaluated with classification report, confusion matrix, ROC-AUC, and average precision.

In [ ]:
def collect_predictions(dataset):
    labels = []
    probs = []
    for inputs, y in dataset:
        pred = model.predict(inputs, verbose=0).reshape(-1)
        probs.extend(pred.tolist())
        labels.extend(y.numpy().reshape(-1).tolist())
    return np.asarray(labels, dtype=int), np.asarray(probs, dtype=float)

val_labels, val_probs = collect_predictions(val_ds)
precision_vals, recall_vals, thresholds = precision_recall_curve(val_labels, val_probs)
f1_scores = 2 * precision_vals[:-1] * recall_vals[:-1] / np.maximum(precision_vals[:-1] + recall_vals[:-1], 1e-8)
best_idx = int(np.nanargmax(f1_scores)) if len(f1_scores) else 0
best_threshold = float(thresholds[best_idx]) if len(thresholds) else 0.50

print(f'Validation-selected threshold: {best_threshold:.3f}')
if len(f1_scores):
    print(f'Validation F1 at threshold: {f1_scores[best_idx]:.3f}')

test_labels, test_probs = collect_predictions(test_ds)
test_pred = (test_probs >= best_threshold).astype(int)

print('\nClassification report on held-out test set:')
print(classification_report(test_labels, test_pred, target_names=['Normal/Non-fractured', 'Fracture/Crack'], digits=4))

roc_auc = roc_auc_score(test_labels, test_probs)
avg_precision = average_precision_score(test_labels, test_probs)
print(f'ROC-AUC: {roc_auc:.4f}')
print(f'Average precision: {avg_precision:.4f}')

cm = confusion_matrix(test_labels, test_pred)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Fracture'], yticklabels=['Normal', 'Fracture'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

fpr, tpr, _ = roc_curve(test_labels, test_probs)
axes[1].plot(fpr, tpr, label=f'ROC-AUC={roc_auc:.3f}')
axes[1].plot([0, 1], [0, 1], linestyle='--', color='gray')
axes[1].set_title('ROC Curve')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

pr_precision, pr_recall, _ = precision_recall_curve(test_labels, test_probs)
axes[2].plot(pr_recall, pr_precision, label=f'AP={avg_precision:.3f}')
axes[2].set_title('Precision-Recall Curve')
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].legend()
axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Interpretation: false negatives deserve special attention because missed fractures are clinically risky.')

## 18. Error Analysis

This cell lists the most confident mistakes. Reviewing these cases is useful for academic discussion because it reveals model limitations and possible data quality issues.

In [ ]:
test_results = test_df.copy().reset_index(drop=True)
test_results['probability'] = test_probs
_test_pred = (test_results['probability'] >= best_threshold).astype(int)
test_results['predicted_label'] = _test_pred.map({0: 'Normal/Non-fractured', 1: 'Fracture/Crack'})
test_results['correct'] = _test_pred.values == test_results['label'].values
test_results['confidence_error'] = np.where(test_results['label'] == 1, 1 - test_results['probability'], test_results['probability'])

mistakes = test_results[~test_results['correct']].sort_values('confidence_error', ascending=False)
print('Number of test mistakes:', len(mistakes))
if len(mistakes):
    display(mistakes[['image_path', 'label_name', 'predicted_label', 'probability', 'age', 'pain_level', 'injury_location']].head(10))
else:
    print('No mistakes at the selected threshold on this test split.')
print('Interpretation: confident false negatives and false positives should be discussed as residual risk.')

## 19. Grad-CAM Localization

Grad-CAM highlights the convolutional image regions that most influenced fracture/crack probability. It is explanatory evidence, not a diagnostic segmentation mask.

In [ ]:
def make_single_example(row):
    image = decode_image(tf.constant(str(row['image_path']))).numpy()
    image_batch = image.reshape(1, IMG_SIZE[0], IMG_SIZE[1], 3).astype('float32')
    tabular_batch = row[FEATURE_NAMES].astype('float32').values.reshape(1, -1)
    return image, image_batch, tabular_batch


def gradcam_heatmap(model, image_batch, tabular_batch, layer_name='last_conv_features'):
    grad_model = tf.keras.models.Model(model.inputs, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model([image_batch, tabular_batch], training=False)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        return None
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    max_value = tf.reduce_max(heatmap)
    if float(max_value) > 0:
        heatmap = heatmap / max_value
    return heatmap.numpy()


def overlay_heatmap(image, heatmap, alpha=0.38):
    image_uint8 = np.uint8(np.clip(image, 0, 255))
    heatmap_resized = cv2.resize(heatmap, (image_uint8.shape[1], image_uint8.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    color_map = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    color_map = cv2.cvtColor(color_map, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(image_uint8, 1 - alpha, color_map, alpha, 0)

sample_rows = test_df.sample(min(4, len(test_df)), random_state=SEED)
plt.figure(figsize=(12, 6))
for idx, (_, row) in enumerate(sample_rows.iterrows(), start=1):
    image, image_batch, tabular_batch = make_single_example(row)
    prob = float(model.predict({'xray_image': image_batch, 'patient_features': tabular_batch}, verbose=0).reshape(-1)[0])
    heatmap = gradcam_heatmap(model, image_batch, tabular_batch)

    ax = plt.subplot(2, len(sample_rows), idx)
    ax.imshow(np.clip(image / 255.0, 0, 1))
    ax.set_title(f"True: {row['label_name']}")
    ax.axis('off')

    ax = plt.subplot(2, len(sample_rows), idx + len(sample_rows))
    if heatmap is not None:
        ax.imshow(overlay_heatmap(image, heatmap))
        ax.set_title(f'Grad-CAM Prob: {prob:.2f}')
    else:
        ax.imshow(np.clip(image / 255.0, 0, 1))
        ax.set_title('Grad-CAM unavailable')
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Interpretation: heatmaps should roughly align with clinically plausible suspicious regions when predictions are correct.')

## 20. Risk Stratification and Automated Report

The report combines CNN probability with patient-level risk factors using a transparent scoring rule. This separation makes the system easier to audit: the neural model produces probability, while clinical factors adjust risk level.

In [ ]:
def estimate_risk(probability, row):
    clinical_points = 0
    clinical_points += int(row['age'] >= 65)
    clinical_points += int(bool(row['diabetes']))
    clinical_points += int(bool(row['osteoporosis']))
    clinical_points += int(bool(row['smoking']))
    clinical_points += int(bool(row['previous_fracture']))
    clinical_points += int(row['pain_level'] >= 8)
    clinical_points += int(row['injury_location'] in {'Hip', 'Spine'})
    clinical_risk = min(clinical_points / 7.0, 1.0)
    risk_score = 100.0 * (0.65 * probability + 0.35 * clinical_risk)
    if risk_score >= 80 or probability >= 0.90:
        level = 'Critical'
    elif risk_score >= 60 or probability >= 0.70:
        level = 'High'
    elif risk_score >= 35 or probability >= 0.45:
        level = 'Moderate'
    else:
        level = 'Low'
    return level, risk_score


def generate_report(row, probability, threshold):
    label = 'Fracture/Crack suspected' if probability >= threshold else 'No fracture pattern detected'
    risk_level, risk_score = estimate_risk(probability, row)
    return f"""Automated X-Ray Fracture & Crack Decision Support Report

Patient: {row.get('patient_name', 'Synthetic Patient')}
Age/Gender: {row['age']} / {row['gender']}
Injury Location: {row['injury_location']}
Pain Level: {row['pain_level']}/10
Diabetes: {row['diabetes']}
Osteoporosis: {row['osteoporosis']}
Smoking: {row['smoking']}
Previous Fracture: {row['previous_fracture']}
Treatment History: {row['treatment_history']}

Model Finding: {label}
Fracture/Crack Probability: {probability:.1%}
Decision Threshold: {threshold:.3f}
Clinical Risk Level: {risk_level}
Risk Score: {risk_score:.1f}/100

Interpretation:
This report fuses CNN image evidence with structured patient risk factors. The Grad-CAM heatmap should be reviewed as explanatory evidence, not as a diagnostic segmentation mask.

Clinical Safety Note:
This academic system is for decision support only and must not replace radiologist or physician review.
"""

example_row = test_df.sample(1, random_state=SEED + 7).iloc[0]
image, image_batch, tabular_batch = make_single_example(example_row)
example_prob = float(model.predict({'xray_image': image_batch, 'patient_features': tabular_batch}, verbose=0).reshape(-1)[0])
print(generate_report(example_row, example_prob, best_threshold))
print('Interpretation: the generated report is concise, reproducible, and clinically cautious.')

## 21. Final Save and Reproducibility Metadata

This cell saves the production model as `fracture_model.keras`, writes deployment metadata, and records the threshold selected from validation data. These files make the Streamlit app deterministic and auditable.

In [ ]:
model.save('fracture_model.keras')
metadata = {
    'project_title': 'Automated Multi-Modal X-Ray Fracture & Crack Reporting System',
    'architecture': f'CNN {CNN_BACKBONE_NAME} image branch + dense tabular branch',
    'supported_backbones': SUPPORTED_BACKBONES,
    'uses_transformer': False,
    'framework': 'TensorFlow/Keras',
    'image_size': list(IMG_SIZE),
    'feature_names': FEATURE_NAMES,
    'threshold': float(best_threshold),
    'label_map': {'0': 'Normal/Non-fractured', '1': 'Fracture/Crack'},
    'selected_data_source': selected_source,
    'coco_annotation_file': str(coco_json_path) if 'coco_json_path' in globals() and coco_json_path is not None else None,
    'test_roc_auc': float(roc_auc),
    'test_average_precision': float(avg_precision),
    'backbone_comparison': backbone_comparison_df.to_dict(orient='records') if 'backbone_comparison_df' in globals() and not backbone_comparison_df.empty else [],
    'clinical_use': 'Educational decision support only; not a standalone diagnostic device.',
}
Path('training_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print('Saved: fracture_model.keras')
print('Saved: training_metadata.json')
print(f'Model size: {Path("fracture_model.keras").stat().st_size / (1024 ** 2):.2f} MB')
print('Interpretation: fracture_model.keras is the trained artifact required by the inference app.')

## 22. In-Notebook Gradio GUI

This is the Colab demonstration interface. It loads the saved `fracture_model.keras`, accepts an X-ray image and patient fields, displays fracture/crack probability, estimates clinical risk, generates a Grad-CAM heatmap, and writes a clinical support report. This GUI is for academic demonstration inside the notebook; production deployment remains separated in the Streamlit app.

In [ ]:
def gui_encode_patient_features(
    age,
    gender,
    diabetes,
    osteoporosis,
    smoking,
    previous_fracture,
    pain_level,
    injury_location,
    treatment_history,
):
    text = (treatment_history or '').lower()
    location = (injury_location or '').lower()
    values = [
        np.clip(float(age), 0.0, 110.0) / 110.0,
        1.0 if gender == 'Male' else 0.0,
        float(bool(diabetes)),
        float(bool(osteoporosis)),
        float(bool(smoking)),
        float(bool(previous_fracture)),
        np.clip(float(pain_level), 1.0, 10.0) / 10.0,
        1.0 if location == 'arm' else 0.0,
        1.0 if location == 'leg' else 0.0,
        1.0 if location == 'spine' else 0.0,
        1.0 if location == 'rib' else 0.0,
        1.0 if location == 'hip' else 0.0,
        1.0 if 'cast' in text else 0.0,
        1.0 if 'surgery' in text or 'operation' in text else 0.0,
        1.0 if 'physio' in text or 'rehab' in text else 0.0,
        1.0 if 'medication' in text or 'analgesic' in text or 'painkiller' in text else 0.0,
        1.0 if not text.strip() or 'none' in text or 'no treatment' in text else 0.0,
    ]
    return np.asarray(values, dtype=np.float32).reshape(1, -1)


def gui_preprocess_image(pil_image):
    image = pil_image.convert('RGB').resize(IMG_SIZE)
    image_array = np.asarray(image, dtype=np.float32)
    return image_array, image_array.reshape(1, IMG_SIZE[0], IMG_SIZE[1], 3)


def gui_predict(
    xray_image,
    patient_name,
    age,
    gender,
    diabetes,
    osteoporosis,
    smoking,
    previous_fracture,
    pain_level,
    injury_location,
    treatment_history,
):
    if xray_image is None:
        return None, 'Please upload an X-ray image first.', '', None

    image_array, image_batch = gui_preprocess_image(xray_image)
    tabular_batch = gui_encode_patient_features(
        age,
        gender,
        diabetes,
        osteoporosis,
        smoking,
        previous_fracture,
        pain_level,
        injury_location,
        treatment_history,
    )

    prediction = gui_model({'xray_image': image_batch, 'patient_features': tabular_batch}, training=False)
    probability = float(np.asarray(prediction).reshape(-1)[0])
    decision = 'Fracture/Crack suspected' if probability >= gui_threshold else 'No fracture pattern detected'

    row = pd.Series({
        'patient_name': patient_name or 'Not provided',
        'age': int(age),
        'gender': gender,
        'diabetes': bool(diabetes),
        'osteoporosis': bool(osteoporosis),
        'smoking': bool(smoking),
        'previous_fracture': bool(previous_fracture),
        'pain_level': int(pain_level),
        'injury_location': injury_location,
        'treatment_history': treatment_history or 'none',
    })
    risk_level, risk_score = estimate_risk(probability, row)

    heatmap = gradcam_heatmap(gui_model, image_batch, tabular_batch)
    if heatmap is not None:
        overlay = overlay_heatmap(image_array, heatmap)
    else:
        overlay = np.uint8(np.clip(image_array, 0, 255))

    summary = (
        f'Prediction: {decision}\n'
        f'Fracture/Crack probability: {probability:.2%}\n'
        f'Decision threshold: {gui_threshold:.3f}\n'
        f'Clinical risk level: {risk_level}\n'
        f'Risk score: {risk_score:.1f}/100'
    )
    report = generate_report(row, probability, gui_threshold)
    report_path = WORK_ROOT / 'gradio_fracture_support_report.txt'
    report_path.write_text(report, encoding='utf-8')
    return overlay, summary, report, str(report_path)


gui_model = tf.keras.models.load_model('fracture_model.keras', compile=False)
gui_threshold = float(metadata.get('threshold', best_threshold))

with gr.Blocks(
    title='X-Ray Fracture & Crack Reporting System',
    css='''
    .gradio-container {max-width: 1200px !important;}
    #report_box textarea {font-family: monospace;}
    '''
) as demo:
    gr.Markdown('# Automated Multi-Modal X-Ray Fracture & Crack Reporting System\nUpload an X-ray image, enter patient factors, then generate a CNN probability, Grad-CAM heatmap, risk level, and clinical support report.')
    with gr.Row():
        with gr.Column(scale=1):
            xray_input = gr.Image(label='X-Ray Image', type='pil')
            patient_name_input = gr.Textbox(label='Patient Name', placeholder='Optional')
            age_input = gr.Slider(1, 110, value=45, step=1, label='Age')
            gender_input = gr.Radio(['Male', 'Female'], value='Male', label='Gender')
            injury_location_input = gr.Dropdown(INJURY_LOCATIONS, value='Arm', label='Injury Location')
            pain_level_input = gr.Slider(1, 10, value=5, step=1, label='Pain Level')
        with gr.Column(scale=1):
            diabetes_input = gr.Checkbox(label='Diabetes')
            osteoporosis_input = gr.Checkbox(label='Osteoporosis')
            smoking_input = gr.Checkbox(label='Smoking')
            previous_fracture_input = gr.Checkbox(label='Previous Fracture')
            treatment_history_input = gr.Textbox(label='Treatment History', placeholder='e.g., none, cast, surgery, medication')
            predict_button = gr.Button('Generate AI Report', variant='primary')
    with gr.Row():
        heatmap_output = gr.Image(label='Grad-CAM Suspicious Region Heatmap', type='numpy')
        summary_output = gr.Textbox(label='Prediction Summary', lines=6)
    report_output = gr.Textbox(label='Generated Clinical Support Report', lines=18, elem_id='report_box')
    report_file_output = gr.File(label='Download Report TXT')

    predict_button.click(
        fn=gui_predict,
        inputs=[
            xray_input,
            patient_name_input,
            age_input,
            gender_input,
            diabetes_input,
            osteoporosis_input,
            smoking_input,
            previous_fracture_input,
            pain_level_input,
            injury_location_input,
            treatment_history_input,
        ],
        outputs=[heatmap_output, summary_output, report_output, report_file_output],
    )

print('Launching Gradio GUI. In Colab, use the public/share link if displayed.')
demo.launch(share=IN_COLAB, debug=False)

## 23. Generate Streamlit Deployment Files

This cell writes the inference-only Streamlit app and dependency file. The app loads `fracture_model.keras`, uses `training_metadata.json` when available, accepts an X-ray and patient fields, shows Grad-CAM, and generates a downloadable report.

In [ ]:
APP_CODE = 'import os\nimport json\nfrom datetime import datetime\n\nimport cv2\nimport numpy as np\nimport streamlit as st\nimport tensorflow as tf\nfrom PIL import Image\n\n\nIMG_SIZE = (224, 224)\nMODEL_PATH = os.getenv("FRACTURE_MODEL_PATH", "fracture_model.keras")\nMODEL_THRESHOLD = float(os.getenv("FRACTURE_THRESHOLD", "0.50"))\nMETADATA_PATH = os.getenv("FRACTURE_METADATA_PATH", "training_metadata.json")\nINJURY_LOCATIONS = ["Arm", "Leg", "Spine", "Rib", "Hip"]\nFEATURE_NAMES = [\n    "age_norm",\n    "gender_male",\n    "diabetes",\n    "osteoporosis",\n    "smoking",\n    "previous_fracture",\n    "pain_norm",\n    "injury_arm",\n    "injury_leg",\n    "injury_spine",\n    "injury_rib",\n    "injury_hip",\n    "treatment_cast",\n    "treatment_surgery",\n    "treatment_physiotherapy",\n    "treatment_medication",\n    "treatment_none",\n]\n\n\nst.set_page_config(\n    page_title="X-Ray Fracture Reporting",\n    layout="wide",\n)\n\n\n@st.cache_resource(show_spinner=False)\ndef load_fracture_model():\n    if not os.path.exists(MODEL_PATH):\n        return None\n    return tf.keras.models.load_model(MODEL_PATH, compile=False)\n\n\n@st.cache_data(show_spinner=False)\ndef load_training_metadata():\n    if not os.path.exists(METADATA_PATH):\n        return {}\n    try:\n        with open(METADATA_PATH, "r", encoding="utf-8") as handle:\n            return json.load(handle)\n    except Exception:\n        return {}\n\n\ndef encode_patient_features(\n    age,\n    gender,\n    diabetes,\n    osteoporosis,\n    smoking,\n    previous_fracture,\n    pain_level,\n    injury_location,\n    treatment_history,\n):\n    text = (treatment_history or "").lower()\n    location = (injury_location or "").lower()\n    age_norm = np.clip(float(age), 0.0, 110.0) / 110.0\n    pain_norm = np.clip(float(pain_level), 1.0, 10.0) / 10.0\n    values = [\n        age_norm,\n        1.0 if gender == "Male" else 0.0,\n        float(bool(diabetes)),\n        float(bool(osteoporosis)),\n        float(bool(smoking)),\n        float(bool(previous_fracture)),\n        pain_norm,\n        1.0 if location == "arm" else 0.0,\n        1.0 if location == "leg" else 0.0,\n        1.0 if location == "spine" else 0.0,\n        1.0 if location == "rib" else 0.0,\n        1.0 if location == "hip" else 0.0,\n        1.0 if "cast" in text else 0.0,\n        1.0 if "surgery" in text or "operation" in text else 0.0,\n        1.0 if "physio" in text or "rehab" in text else 0.0,\n        1.0 if "medication" in text or "analgesic" in text or "painkiller" in text else 0.0,\n        1.0 if not text.strip() or "none" in text or "no treatment" in text else 0.0,\n    ]\n    return np.asarray(values, dtype=np.float32).reshape(1, -1)\n\n\ndef preprocess_xray(image):\n    image = image.convert("RGB").resize(IMG_SIZE)\n    return np.asarray(image, dtype=np.float32).reshape(1, IMG_SIZE[0], IMG_SIZE[1], 3)\n\n\ndef predict_fracture(model, image_batch, tabular_batch):\n    try:\n        pred = model(\n            {"xray_image": image_batch, "patient_features": tabular_batch},\n            training=False,\n        )\n    except Exception:\n        pred = model([image_batch, tabular_batch], training=False)\n    return float(np.asarray(pred).reshape(-1)[0])\n\n\ndef find_gradcam_layer(model):\n    for name in ("last_conv_features", "Conv_1", "out_relu"):\n        try:\n            model.get_layer(name)\n            return name\n        except Exception:\n            pass\n\n    for layer in reversed(model.layers):\n        output_shape = getattr(layer, "output_shape", None)\n        if output_shape is not None and len(output_shape) == 4:\n            return layer.name\n        try:\n            if len(layer.output.shape) == 4:\n                return layer.name\n        except Exception:\n            continue\n    return None\n\n\ndef make_gradcam_overlay(model, pil_image, image_batch, tabular_batch):\n    layer_name = find_gradcam_layer(model)\n    if layer_name is None:\n        return None\n\n    grad_model = tf.keras.models.Model(\n        model.inputs,\n        [model.get_layer(layer_name).output, model.output],\n    )\n    with tf.GradientTape() as tape:\n        conv_outputs, predictions = grad_model([image_batch, tabular_batch], training=False)\n        loss = predictions[:, 0]\n\n    grads = tape.gradient(loss, conv_outputs)\n    if grads is None:\n        return None\n\n    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))\n    conv_outputs = conv_outputs[0]\n    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)\n    heatmap = tf.maximum(heatmap, 0)\n    max_value = tf.reduce_max(heatmap)\n    if float(max_value) > 0:\n        heatmap = heatmap / max_value\n\n    heatmap = heatmap.numpy()\n    original = np.asarray(pil_image.convert("RGB"))\n    heatmap = cv2.resize(heatmap, (original.shape[1], original.shape[0]))\n    heatmap = np.uint8(255 * heatmap)\n    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)\n    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)\n    overlay = cv2.addWeighted(original, 0.62, heatmap, 0.38, 0)\n    return overlay\n\n\ndef estimate_risk(probability, age, diabetes, osteoporosis, smoking, previous_fracture, pain, location):\n    clinical_points = 0\n    clinical_points += int(age >= 65)\n    clinical_points += int(diabetes)\n    clinical_points += int(osteoporosis)\n    clinical_points += int(smoking)\n    clinical_points += int(previous_fracture)\n    clinical_points += int(pain >= 8)\n    clinical_points += int(location in {"Hip", "Spine"})\n    clinical_risk = min(clinical_points / 7.0, 1.0)\n    risk_score = 100.0 * (0.65 * probability + 0.35 * clinical_risk)\n\n    if risk_score >= 80 or probability >= 0.90:\n        level = "Critical"\n    elif risk_score >= 60 or probability >= 0.70:\n        level = "High"\n    elif risk_score >= 35 or probability >= 0.45:\n        level = "Moderate"\n    else:\n        level = "Low"\n    return level, risk_score\n\n\ndef build_report(patient_name, probability, risk_level, risk_score, label, age, gender, location, pain):\n    generated_at = datetime.now().strftime("%Y-%m-%d %H:%M")\n    return f"""Automated X-Ray Fracture & Crack Decision Support Report\n\nGenerated: {generated_at}\nPatient: {patient_name or "Not provided"}\nAge/Gender: {age} / {gender}\nInjury Location: {location}\nPain Level: {pain}/10\n\nModel Finding: {label}\nFracture/Crack Probability: {probability:.1%}\nClinical Risk Level: {risk_level}\nRisk Score: {risk_score:.1f}/100\n\nInterpretation:\nThis report combines a CNN-based X-ray probability estimate with structured\npatient risk factors. The heatmap highlights image regions that most influenced\nthe model output.\n\nClinical Safety Note:\nThis system is for educational decision support only. It is not a standalone\ndiagnostic device and must not replace radiologist or physician review.\n"""\n\n\nst.title("Automated Multi-Modal X-Ray Fracture & Crack Reporting System")\nmetadata = load_training_metadata()\nthreshold = float(metadata.get("threshold", MODEL_THRESHOLD))\n\nmodel = load_fracture_model()\nif model is None:\n    st.error(\n        f"`{MODEL_PATH}` was not found. Train the model in the Colab notebook, "\n        "download `fracture_model.keras`, and place it beside this app."\n    )\n    st.stop()\n\nleft, right = st.columns([1.05, 0.95])\n\nwith left:\n    uploaded_file = st.file_uploader("Upload X-Ray image", type=["png", "jpg", "jpeg"])\n    if uploaded_file:\n        xray_image = Image.open(uploaded_file)\n        st.image(xray_image, caption="Input X-Ray", use_container_width=True)\n\nwith right:\n    patient_name = st.text_input("Patient Name")\n    age = st.number_input("Age", min_value=1, max_value=110, value=45)\n    gender = st.selectbox("Gender", ["Male", "Female"])\n    location = st.selectbox("Injury Location", INJURY_LOCATIONS)\n    pain_level = st.slider("Pain Level", 1, 10, 5)\n    diabetes = st.checkbox("Diabetes")\n    osteoporosis = st.checkbox("Osteoporosis")\n    smoking = st.checkbox("Smoking")\n    previous_fracture = st.checkbox("Previous Fracture")\n    treatment_history = st.text_area("Treatment History", placeholder="e.g., cast, surgery, medication")\n\nif uploaded_file and st.button("Generate Report", type="primary"):\n    image_batch = preprocess_xray(xray_image)\n    tabular_batch = encode_patient_features(\n        age,\n        gender,\n        diabetes,\n        osteoporosis,\n        smoking,\n        previous_fracture,\n        pain_level,\n        location,\n        treatment_history,\n    )\n    probability = predict_fracture(model, image_batch, tabular_batch)\n    label = "Fracture/Crack suspected" if probability >= threshold else "No fracture pattern detected"\n    risk_level, risk_score = estimate_risk(\n        probability,\n        age,\n        diabetes,\n        osteoporosis,\n        smoking,\n        previous_fracture,\n        pain_level,\n        location,\n    )\n\n    result_col, heatmap_col = st.columns(2)\n    with result_col:\n        st.metric("Fracture/Crack Probability", f"{probability:.1%}")\n        st.metric("Clinical Risk Level", risk_level, f"{risk_score:.1f}/100")\n        st.write(label)\n\n    with heatmap_col:\n        overlay = make_gradcam_overlay(model, xray_image, image_batch, tabular_batch)\n        if overlay is not None:\n            st.image(overlay, caption="Grad-CAM Suspicious Region Heatmap", use_container_width=True)\n        else:\n            st.warning("Heatmap could not be generated for this model architecture.")\n\n    report = build_report(\n        patient_name,\n        probability,\n        risk_level,\n        risk_score,\n        label,\n        age,\n        gender,\n        location,\n        pain_level,\n    )\n    st.text_area("Generated Clinical Support Report", report, height=330)\n    st.download_button(\n        "Download Report",\n        data=report,\n        file_name="fracture_support_report.txt",\n        mime="text/plain",\n    )\n'
REQUIREMENTS = 'streamlit>=1.36.0\ntensorflow-cpu==2.16.2\nopencv-python-headless>=4.9.0.80\npillow>=10.3.0\nnumpy>=1.26.0\n'

Path('streamlit_app.py').write_text(APP_CODE, encoding='utf-8')
Path('requirements.txt').write_text(REQUIREMENTS, encoding='utf-8')

print('Saved deployment file: streamlit_app.py')
print('Saved dependency file: requirements.txt')
print('Interpretation: the app is inference-only and contains no training loop.')

## 24. Bundle and Download Artifacts

This final cell creates a clean deployment archive containing only the files needed for Streamlit Cloud. The downloaded FracAtlas dataset is intentionally excluded.

In [ ]:
bundle_path = Path('fracture_streamlit_deployment_bundle.zip')
artifact_files = [
    Path('streamlit_app.py'),
    Path('requirements.txt'),
    Path('fracture_model.keras'),
    Path('training_metadata.json'),
]

with ZipFile(bundle_path, 'w') as zf:
    for file_path in artifact_files:
        if file_path.exists():
            zf.write(file_path, arcname=file_path.name)

print('Created bundle:', bundle_path.resolve())
print(f'Bundle size: {bundle_path.stat().st_size / (1024 ** 2):.2f} MB')
print('Interpretation: upload app.py, requirements.txt, training_metadata.json, and fracture_model.keras to Streamlit Cloud.')

if IN_COLAB:
    from google.colab import files
    files.download(str(bundle_path))

## 25. Suggested Academic Discussion Points

- **CNN backbone choice:** the notebook supports EfficientNetB0, EfficientNetB3, DenseNet121, ResNet50, MobileNetV2, InceptionV3, and Xception. DenseNet121 is a strong default for X-ray work, and the optional comparison cell can justify final model selection.
- **Dataset:** FracAtlas supports fracture classification and localization research, but prospective hospital validation is still required.
- **Synthetic patient factors:** tabular fields are generated because public FracAtlas does not include patient history. This demonstrates multi-modal fusion but does not validate real clinical risk modeling.
- **Imbalance:** report class distribution, class weights, recall, and false negatives.
- **Explainability:** Grad-CAM gives model evidence, not ground-truth segmentation.
- **Deployment boundary:** training happens only in this notebook; Streamlit performs inference only.
- **Safety:** the system is decision support for academic work, not a replacement for clinical judgment.

## 26. Colab Upload Checklist

Before submission or Colab upload:

- Upload only this notebook to Colab; it downloads FracAtlas automatically.
- Use GPU runtime for training.
- Keep `RUN_BACKBONE_COMPARISON = False` for a normal full run, or set it to `True` only when you want the optional backbone comparison experiment.
- The final model artifact is `fracture_model.keras`.
- The in-notebook Gradio GUI runs after training and saving.
- Dataset folders are runtime-only and should not be uploaded to GitHub.